## Install & Importing all important library

In [ ]:
!pip install -q transformers accelerate bitsandbytes plotly pandas sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.8 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown

In [ ]:
import json

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Loading the llama 2 model from huggingface

In [ ]:
model_id = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True  # 4-bit quantization for T4
)

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

<s>[INST] <<SYS>>
You are a helpful data extraction assistant. Return ONLY valid JSON.
<</SYS>> [/INST]

## Creating a prompt from user query in a llama 2 format

In [ ]:
def create_llama2_prompt(user_query):
    return {user_query}

In [ ]:
def query_llama2(prompt, max_new_tokens=500):
    try:
        formatted_prompt = create_llama2_prompt(prompt)
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            top_p=0.9
        )
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract just the assistant's response
        return full_response.split("[/INST]")[-1].strip()
    except Exception as e:
        print(f"Query failed: {str(e)}")
        return None

Convert this to JSON: Convert this to JSON:
Required format:
{{
"metrics": [
{{"name": "str", "value": float, "period": "str", "change": "str"}}
],
"entities": ["str"]
}}

IMPORTANT:
1. Convert all values to numbers (e.g., "$51.3B" → 51300000000)
2. Include ONLY the JSON object
3. No additional text or explanations"""

In [ ]:
def extract_data(text):
    prompt = {text}

    try:
        response = query_llama2(prompt)

        # Handle common response patterns
        json_str = response.replace("```json", "").replace("```", "").strip()
        json_str = json_str[json_str.find('{'):json_str.rfind('}')+1]

        data = json.loads(json_str)

        # Transform different JSON formats to our standard
        if "metrics" not in data:
            # Handle cases where data is nested differently
            transformed = {"metrics": [], "entities": []}
            for key, value in data.items():
                if isinstance(value, dict):  # For nested structures
                    for k, v in value.items():
                        transformed["metrics"].append({
                            "name": k,
                            "value": convert_value(v),
                            "period": "Q3 2024"  # Default if not specified
                        })
                    transformed["entities"].append(key)
            return transformed

        # Convert all values to numbers
        for item in data["metrics"]:
            item["value"] = convert_value(item["value"])

        return data

    except Exception as e:
        print(f"Extraction failed: {str(e)}")
        print("Problematic response:", response)
        return None

def convert_value(value):
    """Convert '$51.3B' → 51300000000.0 or '15%' → 15"""
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        value = value.replace('$', '').replace(',', '')
        if 'B' in value:
            return float(value.replace('B', '')) * 1e9
        elif 'M' in value:
            return float(value.replace('M', '')) * 1e6
        elif '%' in value:
            return float(value.replace('%', ''))
    return float(value)

## Function for data visualization using bar chart and time-series plot

In [ ]:
def visualize_data(data):
    if not data:
        print("No data to visualize")
        return

    try:
        df = pd.DataFrame(data['metrics'])

        # Convert all values to numeric (handles % changes)
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df = df.dropna(subset=['value'])

        # Time-series plot
        if 'period' in df.columns:
            fig = px.line(df, x='period', y='value', color='name',
                         title='Trend Analysis', markers=True)
            fig.update_yaxes(tickprefix="$", tickformat=",.2s")
            fig.show()

        # Bar chart
        fig = px.bar(df, x='name', y='value', text='value',
                    title='Metric Comparison', color='name')
        fig.update_traces(
            texttemplate='%{text:$,.2s}',
            textposition='outside'
        )
        fig.update_yaxes(tickprefix="$")
        fig.show()

    except Exception as e:
        print(f"Visualization failed: {str(e)}")
        print("Problematic data:", data)

Analyze this data and create a business story:
Include:
1. Key trend summary
2. Notable performance highlights
3. Future outlook

In [ ]:
def generate_story(data):
    if not data:
        return "No data available for story generation"

    try:
        prompt = {json.dumps(data, indent=2)}

        response = query_llama2(prompt, max_new_tokens=800)
        return response.split("[/INST]")[-1].strip() if response else "Story generation failed"
    except Exception as e:
        return f"Story generation error: {str(e)}"

## Create a pipeline

In [ ]:
def run_pipeline(input_text):
    print("Extracting data...")
    data = extract_data(input_text)

    if data:
        display(Markdown("### Extracted Data"))
        display(data)

        print("\n Visualizing...")
        visualize_data(data)

        print("\n Generating story...")
        story = generate_story(data)
        display(Markdown("### Data Story"))
        display(Markdown(story))
    else:
        print(" Pipeline failed - check input format")


## Testing with the custome prompt

In [ ]:
test_text = """Samsung Q1 2025 Results:
- Samsung phone revenue: $56.8B (up 3.7% YoY)
- Services: $21.1B (up 12.7%)
- Home applience sales: $10.9B (down 1.1%)
- R&D investment: $1.2B"""

print(" Testing extraction...")
fixed_data = extract_data(test_text)
if fixed_data:
    display(fixed_data)
    print("\n Testing visualization...")
    visualize_data(fixed_data)
else:
    print("Failed to extract data")

 Testing extraction...


{'metrics': [{'name': 'Samsung phone revenue',
   'value': 56800000000.0,
   'period': 'Q3 2024'},
  {'name': 'Services', 'value': 21100000000.0, 'period': 'Q3 2024'},
  {'name': 'Home appliance sales',
   'value': 10900000000.0,
   'period': 'Q3 2024'},
  {'name': 'R&D investment', 'value': 1200000000.0, 'period': 'Q3 2024'}],
 'entities': ['Samsung Q1 2025 Results']}


 Testing visualization...


In [ ]:
test_text_2 = """Google Q2 2024 Earnings:
- Search ads: $42.8B (up 11.3%)
- YouTube ads: $8.4B (up 18.7%)
- Cloud revenue: $9.2B (up 28.4%)
- AI investments: $3.5B"""

if custom_text.strip():
    print("\n Processing custom input...")
    run_pipeline(test_text_2)


 Processing custom input...
Extracting data...


### Extracted Data

{'metrics': [{'name': 'Search ads',
   'value': 42800000000.0,
   'period': 'Q3 2024'},
  {'name': 'YouTube ads', 'value': 8400000000.0, 'period': 'Q3 2024'},
  {'name': 'Cloud revenue', 'value': 9200000000.0, 'period': 'Q3 2024'},
  {'name': 'AI investments', 'value': 3500000000.0, 'period': 'Q3 2024'}],
 'entities': ['Google Q2 2024 Earnings']}


 Visualizing...



 Generating story...


### Data Story

Sure! Here is the valid JSON without explanations:

{
  "metrics": [
    {
      "name": "Search ads",
      "value": 4280000000.0,
      "period": "Q3 2024"
    },
    {
      "name": "YouTube ads",
      "value": 840000000.0,
      "period": "Q3 2024"
    },
    {
      "name": "Cloud revenue",
      "value": 9200000000.0,
      "period": "Q3 2024"
    },
    {
      "name": "AI investments",
      "value": 3500000000.0,
      "period": "Q3 2024"
    }
  ],
  "entities": [
    "Google Q2 2024 Earnings"
  ]
}